# Ch4 Transformer Block 教案

**课程名称：** Transformer Block：搭建现代 LLM 积木

**预计总时长：** 75-85 分钟

**源文件：** `Ch4_Transformer_Block/Ch4_Transformer_Block.ipynb`（共 26 个 Cell，Cell 0-25）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-08:00 | 开场回顾 + 环境准备 | Cell 0-4 | 8 分钟 |
| 08:00-25:00 | 残差连接：梯度的高速公路 | Cell 5-8 | 17 分钟 |
| 25:00-30:00 | 休息 + 回顾 | -- | 5 分钟 |
| 30:00-48:00 | LayerNorm：层归一化（含练习） | Cell 9-11 | 18 分钟 |
| 48:00-60:00 | FFN/MLP：模型的记忆区 | Cell 12-15 | 12 分钟 |
| 60:00-65:00 | 休息 + 回顾 | -- | 5 分钟 |
| 65:00-78:00 | 完整 Transformer Block + 堆叠 | Cell 16-21 | 13 分钟 |
| 78:00-85:00 | 总结 + 练习 + 预告下一章 | Cell 22-25 | 7 分钟 |

---

## 课前准备

- [ ] 确认 PyTorch 已安装（源 notebook 使用 PyTorch 2.10.0+cu128）
- [ ] 确认 matplotlib、numpy 可用
- [ ] 确认中文字体设置正确（Microsoft YaHei / SimHei）
- [ ] 提前运行一遍全部 Cell，确认无报错
- [ ] 确认 `resnet_kaiming.png` 图片文件在 Ch4 目录下
- [ ] 准备白板/画板用于画残差连接和 Block 结构流程图
- [ ] 回顾 Ch3 Self-Attention 内容，准备过渡话术

---

## 第一段：开场回顾 + 环境准备（Cell 0-4）

📍 运行 Cell 0-3（Markdown），Cell 4（代码：环境导入）

⏱ 时间分配：8 分钟（回顾 5 分钟 + 环境 3 分钟）

🎯 本段目标
- 回顾 Ch3 Self-Attention 核心概念，建立连接
- 提出新问题：光有 Attention 还不够，还需要什么？
- 确认运行环境就绪
- 让学生对本章四大组件有全局感

🗣 讲课话术

> 大家好，上一章我们学了 Self-Attention，学会了让每个词"看看"周围的词来动态调整自己的表示。但我问大家一个问题——如果我们把 100 层 Attention 堆在一起，会怎样？
>
> 答案是：训练根本跑不动。梯度在反向传播的时候会越来越小，到第 50 层的时候基本就消失了。就好像你在一条很长的走廊里传话，传了 100 个人，最后一个人听到的可能完全不是原来的意思。
>
> 另外，Attention 本质上是加权求和——它是线性的。只靠线性操作，模型能学到的模式是有限的。
>
> 所以今天我们要学习四个关键组件，把它们和 Attention 组装在一起，形成一个完整的 Transformer Block。大家看 Cell 2 的结构图：输入先经过 LayerNorm 归一化，然后过 Self-Attention，再加上残差连接；接着再 LayerNorm，过 FFN 前馈网络，再加残差连接。这就是一个完整的积木块。
>
> 我们先把环境跑起来。运行 Cell 4。

👀 输出要点
- Cell 4 应输出：`PyTorch version: 2.10.0+cu128`
- 如果版本不同不影响运行，只要 >= 1.9 即可

❓ 预判问题

Q: 为什么不直接加更多 Attention 层就好了？
A: 深层网络存在梯度消失和退化问题——层数越多，训练效果反而越差。这正是残差连接要解决的问题，马上就讲。

Q: Cell 2 里面的结构图中 LayerNorm 放在 Attention 前面还是后面？
A: 这里画的是 Pre-LN（前归一化），是现代 LLM 的做法。原始 Transformer 用的是 Post-LN。第五段我们会详细对比两种方案。

➡️ 转场

> 好，环境没问题。接下来我们从最基础但最重要的组件开始——残差连接。它是深度学习历史上最优雅的发明之一。

---

## 第二段：残差连接——梯度的高速公路（Cell 5-8）

📍 运行 Cell 5（残差连接理论 Markdown）、Cell 6（ResNet 原图）、Cell 7（代码：有/无残差对比）、Cell 8（可视化：信号传播对比图）

⏱ 时间分配：17 分钟（理论 7 分钟 + 代码 5 分钟 + 可视化 5 分钟）

🎯 本段目标
- 理解梯度消失问题的严重性（100 层后梯度仅 2.66e-5）
- 掌握残差连接的数学原理：`output = f(x) + x`，梯度中多了个 "+1"
- 理解"学习残差比学习完整映射更容易"的核心洞察
- 通过代码和可视化验证残差连接的效果

🗣 讲课话术

> 我们先看 Cell 5 的理论。假设你有一个 100 层的网络，没有残差连接。每一层的梯度乘以一个因子，假设这个因子是 0.9——看起来很接近 1 对吧？
>
> 但问题是：0.9 的 10 次方是 0.349，还剩 35% 的梯度。0.9 的 50 次方呢？只有 0.005，不到 1% 了。0.9 的 100 次方？2.66e-5，基本就是零。这就是梯度消失。
>
> 残差连接的解决方案简单到令人惊讶：把输入直接加到输出上。`output = f(x) + x`。就多了这一个 `+ x`。
>
> 为什么这么有效？我们对 `y = f(x) + x` 求导：`dy/dx = df/dx + 1`。看到那个 **+1** 了吗？不管 `df/dx` 有多小，梯度至少是 1。梯度有了一条"高速公路"，可以不经过任何层的变换，直接从最后一层流回第一层。
>
> 还有一个更深刻的洞察——大家看 Cell 5 中"核心洞察"部分。假设某一层的最优映射接近恒等映射，也就是 H(x) 约等于 x。没有残差的话，网络要从随机初始化学出一个恒等映射，这对非线性网络来说并不容易。有残差的话，网络只要把 F(x) 推向零——权重初始化接近零即可。**学残差比学完整映射容易得多。**
>
> 现在运行 Cell 7 看实际效果。我们堆叠 10 层，对比有和没有残差连接的情况。
>
> 看输出！无残差连接的输出范围是 [-0.17, 0.22]——信号几乎被压缩没了。有残差连接的呢？[-14.80, 13.94]——信号保持了正常的幅度。
>
> 再看 Cell 8 的可视化。左图是没有残差的情况：30 层之后，输入 2 在第 30 层变成了几乎看不见的小数。右图有残差：信号不但没有消失，还稳稳地保持住了。
>
> Cell 6 是何恺明 2015 年 ResNet 论文的原图。ResNet 让训练 100 多层的网络成为可能，拿了当年 ImageNet 冠军。同样的思想，被直接用在了 Transformer 里。

👀 输出要点
- Cell 7 关键输出：
  - `无残差连接: 输出范围: [-0.17, 0.22]`
  - `有残差连接: 输出范围: [-14.80, 13.94]`
- Cell 8：两张并排图——左图信号逐层衰减（第 30 层几乎为 0），右图信号保持稳定
- 强调数值对比：无残差信号幅度仅 0.39，有残差信号幅度 28.74，差了约 74 倍

❓ 预判问题

Q: 残差连接要求输入和输出维度相同，如果不同怎么办？
A: 可以加一个 1x1 卷积或线性投影来对齐维度。但在 Transformer 中，Attention 和 FFN 的输出维度和输入维度天然相同（d_model），所以不需要额外处理。

Q: 为什么不用更大的学习率来对抗梯度消失？
A: 大学习率会导致梯度爆炸。残差连接的优雅之处在于它不需要调超参数，结构本身就保证了梯度稳定。

Q: 代码里无残差那组信号缩小了，但为什么没有完全为零？
A: 因为只堆了 10 层，如果堆 100 层会更明显。而且 `nn.Linear` 有 bias 项，不会完全退化到零。

➡️ 转场

> 残差连接解决了梯度消失的问题。但还有另一个问题：每层的输出分布可能差别很大，有的均值 5、方差 100，有的均值 -3、方差 0.01。这种分布漂移会让训练变得困难。怎么办？LayerNorm 归一化。

---

## 休息 + 回顾（第 25-30 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 没有残差连接的深层网络梯度会消失——0.9 的 100 次方只剩 2.66e-5，模型学不动。
2. 残差连接 `output = f(x) + x` 让梯度公式中多了个 "+1"，梯度有了直通高速公路。
3. 代码验证：10 层无残差信号幅度仅 0.39，有残差信号幅度 28.74——差了 74 倍。

**下一段预告：**

> 接下来我们学 LayerNorm，让数据分布变得整齐。这里有一个动手练习——手写 LayerNorm 的四个步骤。

---

## 第三段：LayerNorm——层归一化（Cell 9-11）

📍 运行 Cell 9（LayerNorm 理论 Markdown），Cell 10（代码：手写 LayerNorm + 练习 TODO），Cell 11（可视化：归一化前后分布对比）

⏱ 时间分配：18 分钟（理论 5 分钟 + 练习 8 分钟 + 可视化 5 分钟）

🎯 本段目标
- 理解 LayerNorm 的数学公式：(x - mean) / sqrt(var + eps) * gamma + beta
- 手写 LayerNorm 的四个步骤（Cell 10 含 TODO 练习）
- 理解为什么 Transformer 用 LayerNorm 而不用 BatchNorm
- 了解 RMSNorm 是 LayerNorm 的简化高效版本

🗣 讲课话术

> 大家想象一个场景：你们班考了一次数学和一次英语。数学满分 150，平均分 90；英语满分 100，平均分 70。直接比较两科的分数有意义吗？
>
> 没有，因为它们的尺度不同。归一化就是把不同尺度的数据统一到同一个标准下。LayerNorm 对每个 token 的特征维度做归一化，让均值变成 0、方差变成 1。
>
> 看 Cell 9 的具体数值例子。x = [2, 4, 6, 8]：
> - 均值 = (2+4+6+8)/4 = 5.0
> - 方差 = ((2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2) / 4 = 5.0
> - 标准差 = sqrt(5) 约等于 2.236
> - 归一化：(2-5)/2.236 = -1.342，(4-5)/2.236 = -0.447，(6-5)/2.236 = 0.447，(8-5)/2.236 = 1.342
>
> 最后还有可学习的 gamma 和 beta 参数做仿射变换。初始时 gamma=1、beta=0，所以一开始就是标准归一化。训练过程中，模型会学到最合适的缩放和偏移。
>
> 一个重要的问题：为什么 Transformer 用 LayerNorm 而不用 BatchNorm？三个核心原因：
> 1. 自然语言的句子长度不同，BatchNorm 在 batch 维度上统计，不同位置的统计量含义不一致
> 2. GPT 推理时逐 token 生成，batch size 通常为 1，BatchNorm 直接失效
> 3. Padding token 会污染 BatchNorm 的统计量
>
> 现在大家动手！Cell 10 有四个 TODO，就是 LayerNorm 的四个步骤。给大家几分钟完成。

### 练习：手写 LayerNorm（Cell 10 的 4 个 TODO）

**Hint 节奏：**

**0-2 分钟：** 自己尝试，不给提示。

**2 分钟第一个提示：**
> 四步分别是：(1) `x.mean(dim=-1, keepdim=True)` 计算均值，(2) `x.var(dim=-1, keepdim=True, unbiased=False)` 计算方差（注意 unbiased=False），(3) 归一化公式，(4) 仿射变换。

**4 分钟关键代码：**
```python
mean = x.mean(dim=-1, keepdim=True)
var = x.var(dim=-1, keepdim=True, unbiased=False)
x_norm = (x - mean) / torch.sqrt(var + self.eps)
output = self.gamma * x_norm + self.beta
```

### 常见错误

1. **忘记 `keepdim=True`**：不保持维度会导致广播失败，`x - mean` 的 shape 不匹配
2. **方差用了 `unbiased=True`**（默认值）：LayerNorm 标准实现除以 N 而非 N-1，要显式设置 `unbiased=False`
3. **忘记加 `self.eps`**：虽然大部分时候不加也能跑，但如果某个 token 的特征全部相同（方差为 0），就会除以零
4. **gamma 和 beta 搞反**：gamma 是乘（缩放），beta 是加（偏移），不是反过来

### 验证标准

运行 Cell 10 后应看到：
- 归一化前均值约 `[5.40, 6.09, 6.65, 4.23, 6.51]`（不为 0）
- 归一化前方差约 `[108.48, 95.06, 71.66, 92.18, 98.19]`（远大于 1）
- 归一化后均值接近 `0`（约 1e-8 级别）
- 归一化后方差接近 `1`（约 1.0159）
- 与 PyTorch 官方实现的差异：`0.00000048`（接近 0）

🗣 验证后话术

> 看结果！归一化前均值在 4-7 之间，方差从 72 到 108 差别很大。归一化后呢？均值都是 1e-8 级别——基本就是 0。方差都是 1.0159——非常接近 1。
>
> 更重要的是最后一行：我们手写的 LayerNorm 和 PyTorch 官方 `nn.LayerNorm` 的差异只有 4.8e-7，几乎完全一致。这说明我们的实现是正确的。
>
> 现在运行 Cell 11 看可视化。左图是归一化前的分布——中心在 5 附近，范围很宽。右图是归一化后——中心移到了 0，形状变成了标准的钟形曲线。这就是 LayerNorm 的效果。
>
> 最后提一下 RMSNorm。Cell 9 底部有介绍——RMSNorm 去掉了"减均值"的步骤，只保留缩放。研究发现减均值的收益很小，但去掉它可以省约 10-15% 的计算量。LLaMA、Qwen、Mistral 这些现代模型都用 RMSNorm。

👀 输出要点
- Cell 10 关键输出：
  - 归一化前均值：`tensor([5.3953, 6.0928, 6.6541, 4.2342, 6.5122])`
  - 归一化前方差：`tensor([108.4763, 95.0552, 71.6612, 92.1775, 98.1938])`
  - 归一化后均值：`tensor([-1.4901e-08, -1.1176e-08, ...])` 接近 0
  - 归一化后方差：`tensor([1.0159, 1.0159, ...])` 接近 1
  - 与 PyTorch 差异：`0.00000048`
- Cell 11：两张直方图并排——左图蓝色（均值约 5，标准差约 10），右图绿色（均值约 0，标准差约 1）

❓ 预判问题

Q: 归一化后方差是 1.0159 而不是精确的 1，为什么？
A: 因为 LayerNorm 用 `unbiased=False`（除以 N），但打印方差时默认用 `unbiased=True`（除以 N-1）。N=64 时差异约 1/63 约等于 0.016，所以 1.0 变成了 1.016。

Q: eps 一般设多大？
A: 通常 1e-5。它的作用只是防止除以零，值越小精度越高，但太小可能在半精度训练时溢出。

Q: RMSNorm 和 LayerNorm 性能差多少？
A: 效果几乎一样（差异在噪声范围内），但计算量少 10-15%。在大模型训练中，这个节省是显著的。

➡️ 转场

> 好的，残差连接解决了梯度消失，LayerNorm 解决了分布漂移。接下来的 FFN 前馈网络是 Transformer 里参数量最大的部分——它是模型的"记忆区"。

---

## 第四段：FFN/MLP——模型的记忆区（Cell 12-15）

📍 运行 Cell 12（FFN 理论 Markdown），Cell 13（小提示 Markdown），Cell 14（代码：FeedForward 实现），Cell 15（可视化：GELU vs ReLU）

⏱ 时间分配：12 分钟（理论 5 分钟 + 代码 3 分钟 + 可视化 4 分钟）

🎯 本段目标
- 理解 FFN 的"升维 -> 激活 -> 降维"结构
- 理解为什么扩展 4 倍（信息瓶颈理论）
- 理解 GELU 相比 ReLU 的优势
- 了解 SwiGLU 是现代 LLM 的主流选择

🗣 讲课话术

> Attention 是让词和词之间交流信息，但交流完了之后，每个词还需要"消化"这些信息。FFN 就是干这个的——它对每个 token 位置独立地做一个两层 MLP。
>
> 结构特别简单：先从 d_model 升维到 4 倍的 d_ff，过一个激活函数，再降回 d_model。为什么要升维？
>
> 打个比方：你要整理一堆文件。如果桌面只有 A4 大小，你只能处理几张纸。但如果桌面是 A4 的 4 倍大，你可以把所有文件铺开、分类、整理，最后再叠回去。高维空间提供了更大的"工作台面"，让网络能学习更复杂的变换。
>
> 现在运行 Cell 14。看输出：输入 `[2, 10, 64]`，中间层维度是 `[2, 10, 256]`——4 倍扩展，输出又回到 `[2, 10, 64]`。参数量是 33,088。
>
> 大家注意这个参数量——33,088。等一下我们看完整 Block 的参数量时会发现，FFN 占了大部分参数。在 GPT-3 里，FFN 的参数量占总参数的约 2/3。这就是为什么 FFN 被称为"记忆区"——大量知识都存储在这些权重矩阵里。
>
> 再看激活函数。运行 Cell 15 看 GELU 和 ReLU 的对比图。蓝色的 ReLU 在 x < 0 时直接砍到 0——硬截断，简单粗暴。红色的 GELU 呢？在 x < 0 时有一个小小的负值区域，平滑过渡。
>
> GELU 的核心思想是"软门控"：x 越大，通过越多；x 越小，通过越少；x 在 0 附近时平滑过渡。ReLU 的问题是"死神经元"——如果某个神经元的输入持续为负，梯度永远为零，这个神经元就"死"了，再也不会更新。GELU 在负数区域仍有微小的梯度，避免了这个问题。
>
> 顺便提一下 Cell 12 底部的 SwiGLU——LLaMA、Qwen 等模型用的激活函数。它引入了一个门控机制，效果更好但多了一个投影矩阵，所以扩展比从 4 倍调整为 8/3 倍来保持参数量不变。

👀 输出要点
- Cell 14 关键输出：
  - `输入: torch.Size([2, 10, 64])`
  - `中间层: [2, 10, 256]  (4x扩展)`
  - `输出: torch.Size([2, 10, 64])`
  - `参数量: 33,088`
- Cell 15：GELU（红）vs ReLU（蓝）曲线对比图。GELU 在 x 约 -0.17 处有一个小的负值最低点，ReLU 在 x < 0 严格为 0
- 底部输出：`GELU 在负数区域有小的负值，比 ReLU 更平滑`

❓ 预判问题

Q: 为什么是 4 倍而不是 2 倍或 8 倍？
A: 4 倍是原始 Transformer 论文的选择，是一个实验上效果好的经验值。后续研究（如 Scaling Laws）表明 4 倍是一个不错的平衡点。使用 SwiGLU 的 LLaMA 改为 8/3 倍是因为门控矩阵带来了额外参数。

Q: FFN 对"每个 token 位置独立"是什么意思？
A: 同一层 FFN 对序列中所有 token 共享权重（同一套 W1、W2），但计算是独立的——token 1 的 FFN 输出不依赖 token 2 的输入。这和 Attention 不同，Attention 中每个 token 的输出依赖所有其他 token。

Q: 参数量 33,088 怎么算出来的？
A: W1 是 64x256 = 16,384 个权重 + 256 个 bias = 16,640；W2 是 256x64 = 16,384 + 64 个 bias = 16,448。合计 16,640 + 16,448 = 33,088。

➡️ 转场

> 好了，我们已经学了三个组件：残差连接、LayerNorm、FFN。加上之前学的 Self-Attention，我们拥有了 Transformer Block 的所有零件。休息一下，然后我们把它们组装起来。

---

## 休息 + 回顾（第 60-65 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. LayerNorm 把数据分布标准化——归一化前均值在 4-7，方差在 72-108；归一化后均值约 0，方差约 1。我们手写的实现与 PyTorch 官方差异仅 4.8e-7。
2. FFN 采用"升维 4x -> GELU -> 降维"结构，参数量 33,088（d_model=64 时），占 Block 参数的大部分，是模型的"记忆区"。
3. GELU 比 ReLU 平滑——没有"死神经元"问题；SwiGLU 加了门控机制，是 LLaMA/Qwen 的主流选择。

**下一段预告：**

> 接下来是本章的高潮——把 Attention、LayerNorm、FFN 和残差连接拼装成完整的 Transformer Block，然后堆叠 6 层看看参数量是多少。

---

## 第五段：完整 Transformer Block + 堆叠（Cell 16-21）

📍 运行 Cell 16（Pre-LN vs Post-LN 理论 Markdown），Cell 17（MultiHeadAttention 辅助类），Cell 18（TransformerBlock 完整实现），Cell 19（打印模型结构），Cell 20（堆叠多层 Markdown），Cell 21（6 层 Transformer 代码）

⏱ 时间分配：13 分钟（Pre-LN 理论 4 分钟 + Block 代码 4 分钟 + 堆叠 5 分钟）

🎯 本段目标
- 理解 Pre-LN vs Post-LN 的区别，以及为什么现代 LLM 用 Pre-LN
- 看懂完整 TransformerBlock 的代码结构
- 理解堆叠多层 Block 形成完整 Transformer 的方式
- 计算并理解参数量的组成

🗣 讲课话术

> 组装之前，先搞清一个关键设计选择：LayerNorm 放在哪里？
>
> 原始 Transformer 论文用 Post-LN：先做 Attention，加上残差，**然后**做 LayerNorm。数学上是 `LayerNorm(x + Attention(x))`。
>
> 现代 LLM（GPT-2/3、LLaMA）用 Pre-LN：**先**做 LayerNorm，再做 Attention，最后加残差。数学上是 `x + Attention(LayerNorm(x))`。
>
> 区别在哪里？看 Cell 16 的分析。Post-LN 中，反向传播的梯度必须穿过 LayerNorm 层，LayerNorm 的雅可比矩阵会对梯度进行不稳定的缩放，训练初期容易发散，**必须使用 learning rate warmup**。
>
> Pre-LN 中，残差路径是干净的加法 `x + ...`，梯度可以直接流回，不受 LayerNorm 影响。训练更稳定，甚至不需要 warmup 也能正常训练。
>
> 有趣的是，Cell 16 还提到了 DeepNorm——微软用一个缩放因子 alpha > 1 来放大残差、压制子层输出，在 Post-LN 上成功训练了 1000 层的 Transformer。所以本质不在于哪个更好，而在于如何控制梯度尺度。
>
> 好，现在看代码。Cell 17 是我们的 MultiHeadAttention 类——上一章学过的，这里作为组件直接用。
>
> Cell 18 是今天的重头戏——`TransformerBlock` 类。看 `forward` 方法，只有两行核心代码：
> ```python
> x = x + self.dropout(self.attention(self.ln1(x), mask))  # Attention + 残差
> x = x + self.dropout(self.ffn(self.ln2(x)))              # FFN + 残差
> ```
> 就这两行！先 LN -> Attention -> 残差；再 LN -> FFN -> 残差。非常干净。
>
> 看输出：输入 `[2, 10, 64]`，输出 `[2, 10, 64]`——维度不变。参数量 49,984。
>
> 再运行 Cell 19 看结构：ln1、ln2 两个 LayerNorm（各 64 个参数），attention（MultiHeadAttention），ffn（FeedForward），dropout。
>
> 最后看 Cell 21——堆叠 6 层 Block 形成完整的 Transformer。代码也很简单：`nn.ModuleList` 装 6 个 Block，加一个最终的 `ln_final`。
>
> 看总参数量：300,032。单个 Block 是 49,984，6 个就是 299,904，再加上最终 LayerNorm 的 128 个参数，正好 300,032。
>
> 对比一下真实模型：GPT-2 Small 有 12 层、d_model=768、12 个头，参数量约 1.17 亿。GPT-3 有 96 层、d_model=12288、96 个头，参数量 1750 亿。结构完全一样，只是数字大了几个数量级。

👀 输出要点
- Cell 18 关键输出：
  - `Transformer Block:`
  - `输入: torch.Size([2, 10, 64])`
  - `输出: torch.Size([2, 10, 64])`
  - `参数量: 49,984`
- Cell 19 模型结构输出：
  - `ln1: LayerNorm, shape: torch.Size([64])`
  - `ln2: LayerNorm, shape: torch.Size([64])`
  - `attention: MultiHeadAttention`
  - `ffn: FeedForward`
  - `dropout: Dropout`
- Cell 21 关键输出：
  - `6层 Transformer:`
  - `输入: torch.Size([2, 10, 64])`
  - `输出: torch.Size([2, 10, 64])`
  - `总参数量: 300,032`

❓ 预判问题

Q: 单个 Block 的 49,984 参数怎么组成的？
A: FFN 贡献 33,088（之前算过），Attention 的 W_qkv（64x192）+ bias(192) = 12,480，W_o（64x64）+ bias(64) = 4,160，合计 Attention 16,640。两个 LayerNorm 各 128（64 个 gamma + 64 个 beta）= 256。总计 33,088 + 16,640 + 256 = 49,984。

Q: 最后的 `ln_final` 为什么需要？
A: Pre-LN 架构中每个 Block 开头做 LN，但最后一个 Block 的输出没有经过 LN。加一个 final LN 确保最终输出分布稳定，供后续的分类头或 LM head 使用。

Q: 6 层够吗？真实模型要多少层？
A: 这里只是演示。GPT-2 Small 用 12 层，GPT-2 Large 用 36 层，GPT-3 用 96 层，LLaMA-70B 用 80 层。层数越多，模型能力越强，但训练成本也越高。

Q: Dropout 在推理时会关掉吗？
A: 是的。调用 `model.eval()` 后 Dropout 自动关闭。训练时随机丢弃一部分神经元防止过拟合，推理时使用全部神经元。

➡️ 转场

> 到这里，我们已经从零搭建了一个完整的 Transformer。从最底层的残差连接到最顶层的多层堆叠，每一个组件我们都理解了原理并亲手实现了。最后我们来做一个总结，然后看看课后练习。

---

## 第六段：总结 + 练习 + 下一章预告（Cell 22-25）

📍 浏览 Cell 22（总结 Markdown），Cell 23（Extra 练习），Cell 24（下一步预告），Cell 25（练习空间）

⏱ 时间分配：7 分钟（总结 3 分钟 + 练习布置 2 分钟 + 预告 2 分钟）

🎯 本段目标
- 串联全章四大组件的关系
- 确认学生掌握核心面试题
- 布置课后练习
- 预告 Ch5 GPT 组装

🗣 讲课话术

> 最后让我们把今天学的东西串起来。看 Cell 22 的结构图：
>
> 输入 -> LN -> Attention -> 残差 -> LN -> FFN -> 残差 -> 输出
>
> 四个组件各司其职：
> - **残差连接**：保证梯度能流过深层网络——`output = f(x) + x`，梯度至少为 1
> - **LayerNorm**：稳定数据分布——均值 0、方差 1，让训练更平稳
> - **Attention**：学习词之间的关系——这是 Transformer 的灵魂
> - **FFN**：引入非线性、存储知识——升维 4x -> GELU -> 降维，参数量占大头
>
> Cell 22 还有三道面试题，大家务必课后看：
> 1. Pre-LN vs Post-LN 的区别——重点是梯度路径的干净程度
> 2. FFN 为什么 4 倍扩展——信息瓶颈理论，高维空间提供更大表达能力
> 3. LayerNorm vs BatchNorm——变长序列、batch size=1、padding 污染三个原因
>
> 课后练习在 Cell 23：
> 1. 实现 Post-LN 版本的 Block，对比训练稳定性
> 2. 尝试 2x、4x、8x 不同的 FFN 扩展比，观察效果
> 3. 思考题：为什么 Attention 参数量比 FFN 少，但同样重要？
>
> 下一章我们会加入位置编码（Sinusoidal 和 RoPE），把多个 Transformer Block 堆叠起来，加上 Embedding 层和输出头，组装一个完整的 GPT 模型，并且实现文本生成的采样策略。从零到一，搭出一个能生成文本的小型 GPT！

👀 输出要点
- Cell 22：完整的 ASCII 结构图、关键公式速查表（5 个公式）、3 道面试题
- Cell 23：3 道课后练习
- Cell 24：下一章预告，链接到 Ch5_GPT_Assembly
- Cell 25：空白练习空间

❓ 预判问题

Q: Post-LN 和 Pre-LN 的实现差异大吗？
A: 非常小，只需要把 `self.ln1` 的位置从 Attention 前面移到残差加法后面即可。但训练行为差异显著。

Q: 思考题提示——Attention 参数为什么比 FFN 少？
A: 以 d_model=64 为例，Attention 的 W_qkv 是 64x192、W_o 是 64x64，共约 16K 参数；FFN 的 W1 是 64x256、W2 是 256x64，共约 33K 参数。FFN 参数量大约是 Attention 的 2 倍。但 Attention 的重要性在于它建立了 token 之间的交互——没有 Attention，FFN 只是对每个 token 独立处理，无法理解上下文。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场回顾，运行环境准备 | 0-4 |
| 8 | 残差连接理论 + ResNet 图 | 5-6 |
| 15 | 残差连接代码对比 + 可视化 | 7-8 |
| 25 | **休息 + 回顾** | -- |
| 30 | LayerNorm 理论 | 9 |
| 35 | LayerNorm 练习（TODO 补全） | 10 |
| 43 | LayerNorm 可视化 | 11 |
| 48 | FFN 理论 + 代码 | 12-14 |
| 55 | GELU vs ReLU 可视化 | 15 |
| 60 | **休息 + 回顾** | -- |
| 65 | Pre-LN vs Post-LN 理论 | 16 |
| 69 | TransformerBlock 代码 + 结构 | 17-19 |
| 73 | 堆叠 6 层 Transformer | 20-21 |
| 78 | 总结 + 面试题 + 练习布置 | 22-25 |
| 85 | 结束 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

| 公式名称 | 数学表达 | 说明 |
|:---|:---|:---|
| 残差连接 | y = f(x) + x, dy/dx = df/dx + 1 | 梯度至少为 1 |
| LayerNorm | LN(x) = (x - mean) / sqrt(var + eps) * gamma + beta | 特征维度归一化 |
| RMSNorm | RMSNorm(x) = x / sqrt(sum(x^2)/d) * gamma | 去掉均值中心化 |
| FFN | FFN(x) = GELU(xW1 + b1)W2 + b2 | 升维 4x -> 激活 -> 降维 |
| GELU | GELU(x) = x * Phi(x) | 软门控激活函数 |

### 张量维度速查

| 变量 | 形状 | 来源 Cell |
|:---|:---|:---|
| 残差对比输入 x | [1, 10, 64] | Cell 7 |
| LayerNorm 输入 | [2, 5, 64] | Cell 10 |
| FFN 输入/输出 | [2, 10, 64] | Cell 14 |
| FFN 中间层 | [2, 10, 256] | Cell 14 |
| TransformerBlock 输入/输出 | [2, 10, 64] | Cell 18 |
| 6 层 Transformer 输入/输出 | [2, 10, 64] | Cell 21 |

### 关键数值

| 数值 | 来源 | 含义 |
|:---|:---|:---|
| 无残差输出范围 [-0.17, 0.22] | Cell 7 | 信号被压缩 |
| 有残差输出范围 [-14.80, 13.94] | Cell 7 | 信号保持 |
| 0.9^100 = 2.66e-5 | Cell 5 理论 | 梯度消失量化 |
| 归一化前均值 ~5.40 | Cell 10 | 偏离 0 |
| 归一化后均值 ~1e-8 | Cell 10 | 接近 0 |
| 归一化后方差 ~1.0159 | Cell 10 | 接近 1 |
| 与 PyTorch 差异 4.8e-7 | Cell 10 | 实现正确 |
| FFN 参数量 33,088 | Cell 14 | d=64, d_ff=256 |
| Block 参数量 49,984 | Cell 18 | 含 Attention+FFN+LN |
| 6 层总参数量 300,032 | Cell 21 | 6 x Block + final LN |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** Cell 4 报错 `ModuleNotFoundError: No module named 'torch'`

**应对：**
1. 在终端运行 `pip install torch torchvision`
2. 如果是 conda 环境：`conda install pytorch -c pytorch`
3. 备选：切换到 Google Colab，上传 notebook

### 场景 2：中文字体不显示

**症状：** Cell 8 和 Cell 11 的 matplotlib 图表中文显示为方框

**应对：**
1. Cell 4 已配置了 `plt.rcParams["font.sans-serif"]` 备选字体列表
2. 如果仍有问题：`plt.rcParams['font.sans-serif'] = ['DejaVu Sans']`（牺牲中文显示）
3. 口头补充图表中文标签的含义

### 场景 3：Cell 10 学生卡住（LayerNorm TODO 补全）

**应对：**
1. 源 notebook 的 TODO 后已有参考答案，确认学生是否在看空白版本
2. 先给结构提示："四步：mean -> var -> normalize -> affine"
3. 再给 API 提示："`x.mean(dim=-1, keepdim=True)` 和 `x.var(dim=-1, keepdim=True, unbiased=False)`"
4. 最多 4 分钟后展示完整答案，不要卡在这里太久

### 场景 4：resnet_kaiming.png 图片加载失败

**症状：** Cell 6 显示图片加载错误

**应对：**
1. 确认图片文件在 `Ch4_Transformer_Block/` 目录下
2. 如果缺失：口头描述 ResNet 的结构图（跳跃连接从输入直接到输出）
3. 用白板画一个简单的残差连接示意图

### 场景 5：时间不够

**可跳过的内容（按优先级）：**
1. Cell 6 ResNet 原图（口头提一句即可）- 省 1 分钟
2. Cell 8 信号传播可视化（Cell 7 的数值输出已经足够说明问题）- 省 3 分钟
3. Cell 19 打印模型结构（直接看 Cell 18 的代码也能理解结构）- 省 2 分钟
4. Cell 16 中 DeepNorm 的介绍（进阶内容）- 省 2 分钟

**不可跳过的核心：**
- Cell 7：残差连接代码对比（本章基础概念的实验验证）
- Cell 10：手写 LayerNorm 练习（本章唯一动手环节）
- Cell 14：FFN 实现和参数量（理解模型参数构成）
- Cell 18：TransformerBlock 完整实现（本章最终目标）
- Cell 21：堆叠 6 层（理解真实模型的构建方式）

### 场景 6：学生提出超纲问题

**常见超纲问题及简要回答：**

| 问题 | 简要回答 | 延伸 |
|:---|:---|:---|
| MoE（混合专家）是什么？ | FFN 变成多个"专家"，每个 token 只激活一部分，提高效率 | 后续课程 |
| 为什么不用 BatchNorm？ | 三个原因（已在 Cell 9 讲过） | 课后复习 |
| FlashAttention 是什么？ | 分块计算 Attention，减少内存读写，不改变数学结果 | 后续章节 |
| KV Cache 是什么？ | 推理加速技巧，缓存已计算的 K/V 避免重复计算 | Ch5 会提到 |